In [ ]:
pip install pandas

1. Структура проекта.

In [ ]:
import os
import shutil


os.chdir('/content')

# Удаление старых версий
for old in ['order_analysis', 'order_review']:
    if os.path.exists(old):
        shutil.rmtree(old)

# Создание структуры
PROJECT = 'order_review'
for folder in ['data', 'reports', 'logs', 'src']:
    os.makedirs(os.path.join(PROJECT, folder), exist_ok=True)

# Создание пустого __init__.py
open(os.path.join(PROJECT, 'src', '__init__.py'), 'w').close()

os.chdir(PROJECT)
print('Текущая директория:', os.getcwd())
print('Содержимое:', os.listdir('.'))

Текущая директория: /content/order_review
Содержимое: ['reports', 'data', 'src', 'logs']


In [ ]:
%%writefile requirements.txt
pandas

Writing requirements.txt


In [ ]:
%%writefile config.py
# Конфигурация проекта

DATA_DIR = 'data'
REPORTS_DIR = 'reports'
LOGS_DIR = 'logs'
REPORT_FILE = 'final_report.csv'
LOG_FILE = 'errors.log'
STATUS_COLUMN = 'status'
STATUS_VALUE = 'Delivered'

Writing config.py


In [ ]:
%%writefile src/analyzer.py
import os
import sys
import pandas as pd
import logging


PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import config


class OrderAnalyzer:
    """Класс для анализа заказов из CSV-файлов."""

    def __init__(self,
                 data_dir=None,
                 reports_dir=None,
                 logs_dir=None,
                 status_column=None,
                 status_value=None):

        self.data_dir = data_dir or config.DATA_DIR
        self.reports_dir = reports_dir or config.REPORTS_DIR
        self.logs_dir = logs_dir or config.LOGS_DIR
        self.report_file = config.REPORT_FILE
        self.log_file = config.LOG_FILE
        self.status_column = status_column or config.STATUS_COLUMN
        self.status_value = status_value or config.STATUS_VALUE

        self._setup_logging()

    def _setup_logging(self):
        """Настройка логгера ошибок."""
        os.makedirs(self.logs_dir, exist_ok=True)
        log_path = os.path.join(self.logs_dir, self.log_file)

        logging.basicConfig(
            filename=log_path,
            level=logging.ERROR,
            format='%(asctime)s - %(levelname)s - %(message)s',
            filemode='a',
            force=True
        )
        self.logger = logging.getLogger(__name__)

    def load_file(self, filepath):
        """Загрузка одного CSV-файла."""
        return pd.read_csv(filepath)

    def filter_by_status(self, df):
        """Фильтрация DataFrame по заданному статусу."""
        return df[df[self.status_column] == self.status_value]

    def calculate_metrics(self, df):
        """Расчёт агрегатных метрик."""
        return {
            'total_orders': len(df),
            'total_revenue': df['total_amount'].sum(),
            'avg_order_value': df['total_amount'].mean()
        }

    def process_one_file(self, filepath):
        """Обработка одного файла: загрузка -> фильтрация -> расчёт."""
        filename = os.path.basename(filepath)
        try:
            df = self.load_file(filepath)
            delivered_df = self.filter_by_status(df)
            metrics = self.calculate_metrics(delivered_df)
            metrics['file_name'] = filename
            print(f"  OK {filename}: {metrics['total_orders']} заказов")
            return metrics
        except Exception as e:
            self.logger.error(f"Ошибка обработки {filename}: {e}")
            print(f"  ERR {filename}: {str(e)[:80]}")
            return None

    def process_all_files(self):
        """Оркестрация: проход по всем CSV в папке data/."""
        os.makedirs(self.reports_dir, exist_ok=True)
        results = []

        print(f"\nОбработка файлов из {self.data_dir}/:")
        csv_files = sorted(
            [f for f in os.listdir(self.data_dir) if f.endswith('.csv')]
        )

        for filename in csv_files:
            filepath = os.path.join(self.data_dir, filename)
            metrics = self.process_one_file(filepath)
            if metrics:
                results.append(metrics)

        return pd.DataFrame(results)

    def save_report(self, df):
        """Сохранение итогового отчёта в CSV."""
        report_path = os.path.join(self.reports_dir, self.report_file)
        df.to_csv(report_path, index=False)
        print(f"Отчёт сохранён: {report_path}")

Writing src/analyzer.py


In [ ]:
%%writefile src/analyzer.py
import os
import sys
import pandas as pd
import logging

# Добавляем корень проекта в sys.path, чтобы корректно импортировать config
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import config


class OrderAnalyzer:
    """Класс для анализа заказов из CSV-файлов."""

    def __init__(self,
                 data_dir=None,
                 reports_dir=None,
                 logs_dir=None,
                 status_column=None,
                 status_value=None):
        # Параметры по умолчанию берём из config, но разрешаем переопределение
        self.data_dir = data_dir or config.DATA_DIR
        self.reports_dir = reports_dir or config.REPORTS_DIR
        self.logs_dir = logs_dir or config.LOGS_DIR
        self.report_file = config.REPORT_FILE
        self.log_file = config.LOG_FILE
        self.status_column = status_column or config.STATUS_COLUMN
        self.status_value = status_value or config.STATUS_VALUE

        self._setup_logging()

    def _setup_logging(self):
        """Настройка логгера ошибок."""
        os.makedirs(self.logs_dir, exist_ok=True)
        log_path = os.path.join(self.logs_dir, self.log_file)

        logging.basicConfig(
            filename=log_path,
            level=logging.ERROR,
            format='%(asctime)s - %(levelname)s - %(message)s',
            filemode='a',
            force=True
        )
        self.logger = logging.getLogger(__name__)

    def load_file(self, filepath):
        """Загрузка одного CSV-файла."""
        return pd.read_csv(filepath)

    def filter_by_status(self, df):
        """Фильтрация DataFrame по заданному статусу."""
        return df[df[self.status_column] == self.status_value]

    def calculate_metrics(self, df):
        """Расчёт агрегатных метрик."""
        return {
            'total_orders': len(df),
            'total_revenue': df['total_amount'].sum(),
            'avg_order_value': df['total_amount'].mean()
        }

    def process_one_file(self, filepath):
        """Обработка одного файла: загрузка -> фильтрация -> расчёт."""
        filename = os.path.basename(filepath)
        try:
            df = self.load_file(filepath)
            delivered_df = self.filter_by_status(df)
            metrics = self.calculate_metrics(delivered_df)
            metrics['file_name'] = filename
            print(f"  OK {filename}: {metrics['total_orders']} заказов")
            return metrics
        except Exception as e:
            self.logger.error(f"Ошибка обработки {filename}: {e}")
            print(f"  ERR {filename}: {str(e)[:80]}")
            return None

    def process_all_files(self):
        """Оркестрация: проход по всем CSV в папке data/."""
        os.makedirs(self.reports_dir, exist_ok=True)
        results = []

        print(f"\nОбработка файлов из {self.data_dir}/:")
        csv_files = sorted(
            [f for f in os.listdir(self.data_dir) if f.endswith('.csv')]
        )

        for filename in csv_files:
            filepath = os.path.join(self.data_dir, filename)
            metrics = self.process_one_file(filepath)
            if metrics:
                results.append(metrics)

        return pd.DataFrame(results)

    def save_report(self, df):
        """Сохранение итогового отчёта в CSV."""
        report_path = os.path.join(self.reports_dir, self.report_file)
        df.to_csv(report_path, index=False)
        print(f"Отчёт сохранён: {report_path}")

Overwriting src/analyzer.py


In [ ]:
%%writefile run.py
"""Точка входа в приложение."""
import sys
import os


sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from src.analyzer import OrderAnalyzer


def main():
    # ПОДГОТОВКА
    analyzer = OrderAnalyzer()

    # ДЕЙСТВИЕ
    report_data = analyzer.process_all_files()

    # СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
    if not report_data.empty:
        analyzer.save_report(report_data)
        print("\nАнализ завершён успешно.")
    else:
        print("\nАнализ завершён, но отчёт не создан (нет валидных файлов).")


if __name__ == "__main__":
    main()

Writing run.py


In [ ]:
from google.colab import files
import os

print("Текущая директория:", os.getcwd())
print("Загрузите три файла:")
print("  - order_10000.csv")
print("  - order_100000.csv")
print("  - order_10000_mistake.csv")
print()

uploaded = files.upload()

for name, content in uploaded.items():
    with open(os.path.join('data', name), 'wb') as f:
        f.write(content)

print(f"\nЗагружено файлов: {len(uploaded)}")
print("Содержимое data/:", os.listdir('data'))

Текущая директория: /content/order_review
Загрузите три файла:
  - order_10000.csv
  - order_100000.csv
  - order_10000_mistake.csv



Saving order_10000_mistake.csv to order_10000_mistake.csv
Saving order_10000.csv to order_10000.csv
Saving order_100000.csv to order_100000.csv

Загружено файлов: 3
Содержимое data/: ['order_100000.csv', 'order_10000.csv', 'order_10000_mistake.csv']


In [ ]:
!python run.py


Обработка файлов из data/:
  OK order_10000.csv: 1417 заказов
  OK order_100000.csv: 14287 заказов
  ERR order_10000_mistake.csv: 'utf-8' codec can't decode byte 0xed in position 4167: invalid continuation byte
Отчёт сохранён: reports/final_report.csv

Анализ завершён успешно.


In [ ]:
import pandas as pd
import os
from google.colab import files

print("Структура проекта")
for root, dirs, files_list in os.walk('.'):
    level = root.replace('.', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files_list:
        print(f'{subindent}{file}')

print("\n" + "=" * 50)
print("reports/final_report.csv")
if os.path.exists('reports/final_report.csv'):
    display(pd.read_csv('reports/final_report.csv'))
else:
    print("Файл не создан")

print("\n" + "=" * 50)
print("logs/errors.log")
if os.path.exists('logs/errors.log') and os.path.getsize('logs/errors.log') > 0:
    with open('logs/errors.log', 'r') as f:
        print(f.read())
else:
    print("(ошибок нет или файл пустой)")

print("\n" + "=" * 50)
print("СКАЧИВАНИЕ РЕЗУЛЬТАТОВ:")
if os.path.exists('reports/final_report.csv'):
    files.download('reports/final_report.csv')
if os.path.exists('logs/errors.log'):
    files.download('logs/errors.log')

=== СТРУКТУРА ПРОЕКТА ===
./
  requirements.txt
  order_100000.csv
  order_10000.csv
  config.py
  order_10000_mistake.csv
  run.py
  reports/
    final_report.csv
  data/
    order_100000.csv
    order_10000.csv
    order_10000_mistake.csv
  src/
    analyzer.py
    __init__.py
    __pycache__/
      analyzer.cpython-312.pyc
      __init__.cpython-312.pyc
  __pycache__/
    config.cpython-312.pyc
  logs/
    errors.log

=== reports/final_report.csv ===


,total_orders,total_revenue,avg_order_value,file_name
0,1417,1069477.83,754.747939,order_10000.csv
1,14287,10874402.61,761.139680,order_100000.csv



=== logs/errors.log ===
2026-06-27 21:15:08,006 - ERROR - Ошибка обработки order_10000_mistake.csv: 'utf-8' codec can't decode byte 0xed in position 4167: invalid continuation byte


СКАЧИВАНИЕ РЕЗУЛЬТАТОВ:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>